# Session 1: a fast stand-in for a slow simulation
**A. Real-world problem.** Some simulations are slow: a combustion engineer may need the answer
for thousands of fuel mixtures, and each run takes minutes. A *surrogate* is a cheap stand-in:
run the slow simulation for a modest number of well-chosen inputs, then learn the pattern
"inputs in, answer out" and use the learned pattern for every other input.

**Plan.** (1) a slow simulation with one input, (2) a surrogate learned from N runs, (3) measure
the error against the true answer, (4) ask how many runs are enough.

**Still a toy.** The "simulation" here is a single decaying substance, so you can see the
whole picture. Sessions 2 and 3 use a genuinely stiff chemical system.

In [ ]:
import pathlib
import time

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

T_END = 1.0                 # the simulation runs for 1 second
K_LO, K_HI = 0.0, 5.0       # input range: decay rate k in 1/s

## Step 1: the slow simulation
A substance decays at rate `k` (units 1/s): `dy/dt = -k y`, starting from `y = 1` (100 percent).
The simulation takes many tiny steps (explicit Euler: `y <- y - k y dt`). The *input* is `k`; the
*answer* is the fraction left after 1 second. The exact answer is `exp(-k)`, which lets us judge
every surrogate.

In [ ]:
def slow_simulation(k, steps=20000):
    """Fraction left after T_END seconds, by many small Euler steps."""
    dt = T_END / steps
    y = 1.0
    for _ in range(steps):
        y = y - k * y * dt
    return y


def exact_answer(k):
    return np.exp(-np.asarray(k, dtype=float) * T_END)

## Step 2: the surrogate = connect the dots
Run the slow simulation for `n` evenly spaced inputs (the *training runs*). The surrogate answers
any new input by drawing straight lines between neighbouring training runs (`np.interp`). This is
the simplest "learned" stand-in; Sessions 2 and 3 replace it by fitted formulas and a neural network.

In [ ]:
def training_runs(n):
    k = np.linspace(K_LO, K_HI, n)
    y = np.array([slow_simulation(v) for v in k])
    return k, y


def make_surrogate(n):
    k, y = training_runs(n)
    return lambda q: np.interp(q, k, y)


def max_error(n, n_test=501):
    """Largest absolute error over a fine grid of test inputs (the fraction left is between 0 and 1)."""
    q = np.linspace(K_LO, K_HI, n_test)
    s = make_surrogate(n)
    return float(np.max(np.abs(s(q) - exact_answer(q))))


def runs_needed(target, n_max=200):
    """Smallest number of training runs whose worst-case error is below `target`."""
    for n in range(2, n_max + 1):
        if max_error(n) < target:
            return n
    return None

## Step 3: cost and error
Building the surrogate costs `n` slow runs once. Every later answer costs almost nothing.

In [ ]:
def time_per_call(fn, repeats):
    t0 = time.perf_counter()
    for i in range(repeats):
        fn(0.5 + 4.0 * (i % 7) / 7.0)
    return (time.perf_counter() - t0) / repeats

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    q = np.linspace(K_LO, K_HI, 501)
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
    for n, col in ((3, "tab:red"), (6, "tab:orange"), (18, "tab:green")):
        k, y = training_runs(n)
        axes[0].plot(q, make_surrogate(n)(q), color=col, label=f"surrogate, {n} runs")
        axes[0].plot(k, y, "o", color=col, ms=4)
        axes[1].semilogy(q, np.abs(make_surrogate(n)(q) - exact_answer(q)) + 1e-12, color=col, label=f"{n} runs")
        print(f"{n:3d} training runs: worst-case error {max_error(n):.4f}")
    axes[0].plot(q, exact_answer(q), "k--", label="true answer exp(-k)")
    axes[0].set_xlabel("decay rate k (1/s)")
    axes[0].set_ylabel("fraction left after 1 s")
    axes[0].set_title("Surrogate from N runs (dots)")
    axes[1].set_xlabel("decay rate k (1/s)")
    axes[1].set_ylabel("absolute error (fraction)")
    axes[1].set_title("Error of the surrogate")
    axes[1].set_ylim(1e-6, 1)
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "surrogate_fit.png", dpi=120)

    ns = np.arange(2, 41)
    errs = [max_error(int(n)) for n in ns]
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.semilogy(ns, errs, "o-", ms=4)
    ax2.axhline(0.01, color="red", ls="--", label="target 0.01")
    ax2.set_xlabel("number of training runs N")
    ax2.set_ylabel("worst-case error (fraction)")
    ax2.set_title("More runs, smaller error")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "error_vs_runs.png", dpi=120)
    print("runs needed for error below 0.01:", runs_needed(0.01))
    t_sim = time_per_call(slow_simulation, 20)
    s = make_surrogate(18)
    t_sur = time_per_call(s, 2000)
    print(f"one slow run {t_sim * 1e3:.2f} ms, one surrogate answer {t_sur * 1e6:.1f} us, ratio {t_sim / t_sur:.0f}")

## Where this stops
Connecting dots needs many runs when there are several inputs (a grid of 18 values per input
needs 324 runs for two inputs and 5832 for three). Session 2 fixes this with fitted formulas, and
applies it to a stiff chemical system, where a fast simulation is hard to build in the first place.

## Coding tasks
1. **Easy.** Run `python start.py run lesson.py`-style: call `max_error(12)` and `max_error(24)` in a short script and explain why the second is more than 3 times smaller.
2. **Stretch.** Change the training inputs to be unevenly spaced (more runs near k = 0) and check whether the worst-case error for 9 runs falls below the evenly spaced value.
3. **Explain.** Explain, in your own words, why the surrogate's answer at k = 6 is wrong and why it gives no warning.

## Assessment
- **A. Conceptual.** Why can a surrogate built from runs at k = 0 to 5 not be trusted at k = 6? What does it actually answer there?
- **B. Numerical.** A surrogate is built from 3 runs (k = 0, 2.5, 5). Find its answer and its error at k = 1.25.
- **C. Graph and simulation interpretation.** In the error graph, where is the error largest, and why? What happens to that peak when you add runs?
- **D. Design and optimisation challenge.** Each slow run takes 2 minutes. Design: how many runs do you need for a worst-case error below 0.01, and below 0.001? What does the tighter target cost in time? What changes if there are two inputs on a grid with the same spacing?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("the slow simulation agrees with exp(-k): k = 1 gives 0.36787 and k = 5 is within 1e-4",
                   abs(L.slow_simulation(1.0) - np.exp(-1.0)) < 1e-4 and abs(L.slow_simulation(5.0) - np.exp(-5.0)) < 1e-4))
    s6 = L.make_surrogate(6)
    hand = (np.exp(-2.0) + np.exp(-3.0)) / 2
    checks.append((f"worked example: with 6 runs (k = 0, 1, ..., 5) the surrogate at k = 2.5 is (0.1353 + 0.0498) / 2 = 0.0926, true 0.0821, error 0.0105 (got {float(s6(2.5)):.4f})",
                   abs(float(s6(2.5)) - hand) < 1e-4 and abs(hand - 0.0926) < 5e-5
                   and abs(float(L.exact_answer(2.5)) - 0.0821) < 5e-5 and abs(abs(float(s6(2.5)) - float(L.exact_answer(2.5))) - 0.0105) < 5e-4))
    k6, y6 = L.training_runs(6)
    checks.append(("the surrogate reproduces its own training runs exactly", np.allclose(s6(k6), y6)))
    checks.append(("worst-case error shrinks with more runs: 3 runs 0.265, 6 runs 0.078, 18 runs 0.0094",
                   abs(L.max_error(3) - 0.265) < 0.002 and abs(L.max_error(6) - 0.0779) < 0.001 and abs(L.max_error(18) - 0.00935) < 0.0003
                   and L.max_error(3) > L.max_error(6) > L.max_error(18)))
    checks.append(("design: 18 runs are the fewest that give an error below 0.01 (17 runs give 0.0105)",
                   L.runs_needed(0.01) == 18 and L.max_error(17) > 0.01 > L.max_error(18)))
    checks.append(("a tenfold tighter target costs about three times as many runs: error below 0.001 needs 56",
                   L.runs_needed(0.001) == 56))
    checks.append(("edge: with only 2 runs the surrogate is one straight line and the worst error is 0.48",
                   abs(L.max_error(2) - 0.480) < 0.002))
    checks.append(("limit: outside the trained range np.interp just repeats the end value, so k = 6 gives 0.0067 instead of 0.0025",
                   abs(float(L.make_surrogate(18)(6.0)) - np.exp(-5.0)) < 1e-4 and abs(float(L.exact_answer(6.0)) - 0.00248) < 1e-4))
    checks.append(("a grid of 18 values per input needs 18 x 18 = 324 runs for two inputs and 5832 for three", 18 ** 2 == 324 and 18 ** 3 == 5832))
    t_sim, t_sur = L.time_per_call(L.slow_simulation, 10), L.time_per_call(L.make_surrogate(18), 1000)
    checks.append((f"speed (illustrative): one surrogate answer is more than 100 times faster than one slow run (ratio {t_sim / t_sur:.0f})",
                   t_sim / t_sur > 100))
    s3 = L.make_surrogate(3)
    est, tru = float(s3(1.25)), float(L.exact_answer(1.25))
    checks.append((f"3 runs at k = 1.25: surrogate {est:.3f}, true {tru:.4f}, error {est - tru:.3f} (nearly 90 percent of the truth)",
                   abs(est - 0.541) < 0.001 and abs(tru - 0.2865) < 5e-5 and abs(est - tru - 0.255) < 0.002 and 0.85 < (est - tru) / tru < 0.9))
    e6 = abs(float(s6(2.5)) - float(L.exact_answer(2.5)))
    checks.append(("6 runs at k = 2.5: the error 0.0105 is about 13 percent of the true answer",
                   0.12 < e6 / float(L.exact_answer(2.5)) < 0.135))
    checks.append(("at k = 6 the surrogate (0.0067) is more than twice the truth (0.0025)",
                   float(L.make_surrogate(18)(6.0)) > 2 * float(L.exact_answer(6.0))))
    checks.append(("run costs at 2 minutes per run: 18 runs 36 min, 56 runs 112 min, 324 runs 648 min",
                   (2 * 18, 2 * 56, 2 * 324) == (36, 112, 648)))
    checks.append(("exercise: 24 runs give an error more than 3 times smaller than 12 runs", L.max_error(12) > 3 * L.max_error(24)))
    q = np.linspace(L.K_LO, L.K_HI, 501)
    err = np.abs(L.make_surrogate(6)(q) - L.exact_answer(q))
    checks.append(("the error of the 6-run surrogate peaks at small k (below 0.5), where exp(-k) bends most", q[int(np.argmax(err))] < 0.5))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
